## Creating a remote function to test RemoteFunctionStep

In [ ]:
import mlrun

project = mlrun.get_or_create_project("test-notebooks", user_project=True)

In [ ]:
%%writefile remote_function.py

def handler(context, event):
    print("Hello World !")
    print(f"event is : {event}")
    event.body['hey'] = "ho"
    print(f"event is : {event}")
    return event

In [ ]:
fn = project.set_function(name='remote-function', kind="remote", func="remote_function.py")

In [ ]:
address = project.deploy_function(fn)

In [ ]:
fn.invoke("", body={"Iguazio": "MLRun"})

## Creating a serving graph with the new created function as a RemoteFunctionStep

In [ ]:
from mlrun.serving.remote import RemoteFunctionStep


step = RemoteFunctionStep(fn=fn.metadata.name, project_name=project.name)

main_fn = mlrun.new_function(name="test-nuclio-remote-step", kind="serving")

flow = main_fn.set_topology("flow", engine="async", exist_ok=True)

flow.to(step).respond()

### Testing Locally

In [ ]:
server = main_fn.to_mock_server()

try:
    resp = server.test(body={"Iguazio": "MLRun"})
except Exception as e:
    raise e
finally:
    server.wait_for_completion()
assert resp['body'] == {"Iguazio": "MLRun", 'hey': 'ho'}

### Testing Remotely

In [ ]:
main_address = main_fn.deploy()

In [ ]:
resp = main_fn.invoke("/", body={"Iguazio": "MLRun"})

In [ ]:
try:
    resp = main_fn.invoke("/", body={"Iguazio": "MLRun"})
except Exception as e:
    raise e
assert resp['body'] == {"Iguazio": "MLRun", 'hey': 'ho'}